# Identify analyses to rerun

Build the full set of country/analysis combinations that should be used, drop those already usable on disk, and write the remainder to `data/config/rerun_pairs.json` for `scripts/massive/jtrauer/rerun_countries.py`.

A combination is in scope if it was requested (`get_analyses_for_country`) and was not skipped (no scaler data). Complete means `store_outputs` finished (`updates.h5` present).

`59746206` trumps `59597639` wherever both exist, because the methods in the later job are more recent. A combination is already available if it is complete in either job; when both are complete, the `59746206` copy is the one to use. Remaining work is requested combinations that are complete in neither folder.

The inventory cell reports how much of each job is actually on disk. If a folder is only a partial copy, combinations that finished remotely but are missing locally will be treated as still needed. Analysis directories under `59597639` that already have a complete `59746206` copy are superseded and can be deleted to free space.


In [1]:
import json
import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH
from emu_renewal.run import get_analyses_for_country

In [2]:
NEWER_RUN = "59746206"
OLDER_RUN = "59597639"
job_ids = [NEWER_RUN, OLDER_RUN]  # later methods take precedence
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))


def job_inventory(run_id):
    run_path = OUTPUTS_PATH / run_id
    present = [p.name for p in run_path.iterdir() if p.is_dir()] if run_path.exists() else []
    n_complete = sum(
        (run_path / iso3 / analysis / "updates.h5").exists()
        for iso3 in present
        for analysis in ANALYSIS_TYPES
    )
    return {"countries on disk": len(present), "complete analyses": n_complete}


pd.DataFrame({job: job_inventory(job) for job in job_ids})

,59746206,59597639
countries on disk,80,111
complete analyses,229,405


In [3]:
def classify_analysis(iso3, analysis, run_path, log_text):
    """Match run.py: complete if store_outputs finished, skipped if ScalerException."""
    if (run_path / iso3 / analysis / "updates.h5").exists():
        return "complete"
    if log_text is None:
        return "no log"
    if f"{analysis} data not available" in log_text:
        return "skipped"
    return "not run"


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

59746206                                          \
              no_scaling oxcgrt_floored oxcgrt_independent g_mob   
complete               1             65                 61    26   
no log                46             46                 46    44   
not requested          0              0                  0     3   
not run               79             15                 19    53   
skipped                0              0                  0     0   

                                                 59597639                 \
              fb_visited_mob fb_singletile_mob no_scaling oxcgrt_floored   
complete                  35                41        109            107   
no log                    44                44         16             16   
not requested              3                 3          0              0   
not run                   44                38          1              3   
skipped                    0                 0          0              0   

                                                                         
              oxcgrt_independent g_mob fb_visited_mob fb_singletile_mob  
complete                      79    49             35                26  
no log                        17    16             16                16  
not requested                  0     3              3                 3  
not run                       30    47             63                72  
skipped                        0    11              9                 9

In [4]:
def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

skipped = (statuses[NEWER_RUN] == "skipped") | (statuses[OLDER_RUN] == "skipped")
complete_newer = statuses[NEWER_RUN] == "complete"
complete_older = statuses[OLDER_RUN] == "complete"
available = complete_newer | complete_older
need = requested & ~skipped & ~available
rerun_pairs = sorted(pairs_from_mask(need))

pd.Series(
    {
        "requested": int(requested.to_numpy().sum()),
        "skipped": int((requested & skipped).to_numpy().sum()),
        "available from 59746206": int((requested & complete_newer).to_numpy().sum()),
        "available from 59597639 only": int(
            (requested & complete_older & ~complete_newer).to_numpy().sum()
        ),
        "remaining": len(rerun_pairs),
    }
)

requested                       747
skipped                          29
available from 59746206         229
available from 59597639 only    309
remaining                       180
dtype: int64

In [5]:
pd.Series([analysis for _, analysis in rerun_pairs]).value_counts()

fb_singletile_mob     47
fb_visited_mob        44
g_mob                 37
oxcgrt_independent    22
no_scaling            17
oxcgrt_floored        13
Name: count, dtype: int64

In [6]:
newer_path = OUTPUTS_PATH / NEWER_RUN
older_path = OUTPUTS_PATH / OLDER_RUN
superseded, keep = [], []
for iso3 in countries:
    for analysis in ANALYSIS_TYPES:
        older_dir = older_path / iso3 / analysis
        if not older_dir.is_dir():
            continue
        rel = f"{iso3}/{analysis}"
        if (newer_path / iso3 / analysis / "updates.h5").exists():
            superseded.append(rel)
        elif (older_dir / "updates.h5").exists():
            keep.append(rel)

(OUTPUTS_PATH / "superseded_original_dirs.txt").write_text("\n".join(superseded) + "\n")
(OUTPUTS_PATH / "keep_from_original.txt").write_text("\n".join(keep) + "\n")
pd.Series(
    {
        "superseded 59597639 dirs (already complete in 59746206)": len(superseded),
        "keep from 59597639 (not in 59746206)": len(keep),
    }
)

superseded 59597639 dirs (already complete in 59746206)     96
keep from 59597639 (not in 59746206)                       309
dtype: int64

In [7]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))